# 09 - Validate Bronze

            Verifies Delta format, append-only history, required lineage
            metadata, and reconciliation between Bronze, quarantine, and DQ
            metrics.

In [ ]:
import re

from pyspark.sql import functions as F

dbutils.widgets.text("catalog", "abc_retail", "Target catalog")
catalog = dbutils.widgets.get("catalog").strip()

if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", catalog):
    raise ValueError(f"Invalid catalog identifier: {catalog!r}")

bronze_tables = {
    "clickstream_events": f"{catalog}.bronze.bronze_clickstream",
    "orders": f"{catalog}.bronze.bronze_orders",
    "customers": f"{catalog}.bronze.bronze_customers",
}
quarantine_tables = {
    "clickstream_events": f"{catalog}.quarantine.clickstream_errors",
    "orders": f"{catalog}.quarantine.orders_errors",
    "customers": f"{catalog}.quarantine.customers_errors",
}
required_metadata = {"ingest_ts", "source_file", "batch_id", "load_date", "pipeline_run_id"}

validation = []
for source_name, table_name in bronze_tables.items():
    exists = spark.catalog.tableExists(table_name)
    detail = spark.sql(f"DESCRIBE DETAIL {table_name}").first() if exists else None
    columns = set(spark.table(table_name).columns) if exists else set()
    null_metadata = (
        spark.table(table_name)
        .filter(
            F.col("ingest_ts").isNull()
            | F.col("source_file").isNull()
            | F.col("batch_id").isNull()
            | F.col("load_date").isNull()
        )
        .count()
        if exists
        else None
    )
    forbidden_ops = (
        spark.sql(f"DESCRIBE HISTORY {table_name}")
        .filter(F.col("operation").isin("UPDATE", "DELETE", "MERGE"))
        .count()
        if exists
        else None
    )
    validation.append(
        {
            "source_name": source_name,
            "table_name": table_name,
            "exists": exists,
            "format": detail["format"] if detail else None,
            "append_only_property": (
                detail["properties"].get("delta.appendOnly") if detail else None
            ),
            "missing_metadata_columns": ",".join(sorted(required_metadata - columns)),
            "null_metadata_rows": null_metadata,
            "forbidden_history_operations": forbidden_ops,
        }
    )

result = spark.createDataFrame(validation)
result.display()

failed = result.filter(
    (~F.col("exists"))
    | (F.col("format") != "delta")
    | (F.col("append_only_property") != "true")
    | (F.col("missing_metadata_columns") != "")
    | (F.col("null_metadata_rows") != 0)
    | (F.col("forbidden_history_operations") != 0)
).count()
if failed:
    raise RuntimeError(f"Bronze structural validation failed for {failed} source(s)")

reconciliation = []
for source_name, bronze_table in bronze_tables.items():
    bronze_count = spark.table(bronze_table).count()
    quarantine_count = (
        spark.table(quarantine_tables[source_name])
        .filter(F.col("error_stage") == "bronze")
        .count()
    )
    dq_count = (
        spark.table(f"{catalog}.control.data_quality_results")
        .filter(
            (F.col("source_name") == source_name)
            & (F.col("pipeline_stage") == "bronze")
        )
        .agg(F.coalesce(F.sum("record_count"), F.lit(0)).alias("count"))
        .first()["count"]
    )
    reconciliation.append(
        {
            "source_name": source_name,
            "bronze_count": bronze_count,
            "quarantine_count": quarantine_count,
            "dq_total_count": int(dq_count),
            "reconciled": bronze_count + quarantine_count == int(dq_count),
        }
    )

reconciliation_df = spark.createDataFrame(reconciliation)
reconciliation_df.display()
if reconciliation_df.filter(~F.col("reconciled")).count():
    raise RuntimeError("Bronze valid/error reconciliation failed")

orders_formats = {
    row.source_format
    for row in spark.table(bronze_tables["orders"])
    .select("source_format")
    .distinct()
    .collect()
}
customers_formats = {
    row.source_format
    for row in spark.table(bronze_tables["customers"])
    .select("source_format")
    .distinct()
    .collect()
}
if not {"json", "parquet"}.issubset(orders_formats):
    raise RuntimeError(f"Orders formats missing: {orders_formats}")
if not {"csv", "json"}.issubset(customers_formats):
    raise RuntimeError(f"Customer formats missing: {customers_formats}")

schema_versions = (
    spark.table(f"{catalog}.control.schema_change_log")
    .filter(F.col("source_name") == "clickstream_events")
    .select("schema_version")
    .distinct()
    .count()
)
if schema_versions == 0:
    raise RuntimeError("No clickstream schema version was logged")

processed_sources = {
    row.source_name
    for row in spark.table(f"{catalog}.control.processed_files")
    .filter(F.col("status") == "SUCCESS")
    .select("source_name")
    .distinct()
    .collect()
}
if not set(bronze_tables).issubset(processed_sources):
    raise RuntimeError(f"Processed-file ledger is missing sources: {processed_sources}")

print("Bronze validation passed")